# LeetCode 839 — Similar String Groups bằng Python

- Link: https://leetcode.com/problems/similar-string-groups/
- Độ khó: **Hard**
- Chủ đề: Đồ thị ẩn, Thành phần liên thông, DFS/BFS, Union-Find (DSU), Chuỗi

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numSimilarGroups(self, strs: List[str]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Hai chuỗi `X`, `Y` gọi là **tương tự** nếu chúng bằng nhau, hoặc có thể biến `X` thành `Y` bằng **đúng một** lần hoán đổi hai ký tự (ở hai vị trí khác nhau). Các chuỗi tạo thành **nhóm** theo quan hệ "tương tự" lan truyền: `"tars"` ~ `"rats"` ~ `"arts"` cùng nhóm dù `"tars"` và `"arts"` không trực tiếp tương tự. Đếm số nhóm.

Đây là bài **đếm thành phần liên thông trên một đồ thị ẩn**: đỉnh là các chuỗi, cạnh nối hai chuỗi tương tự. Không ai cho sẵn danh sách cạnh — ta phải **tự kiểm tra** từng cặp (hoặc tự sinh hàng xóm). Chọn cách nào tùy vào số chuỗi `n` và độ dài `m`.

Sau notebook này bạn sẽ:

1. Viết hàm kiểm tra "tương tự" trong `O(m)` dựa trên đếm vị trí khác nhau (tận dụng giả thiết các chuỗi là hoán vị của nhau).
2. Đếm thành phần liên thông bằng DFS, BFS và Union-Find.
3. Hiểu cách **sinh hàng xóm** bằng mọi phép hoán đổi khi `n` lớn và `m` nhỏ.
4. So sánh hai chiến lược `O(n²·m)` và `O(n·m³)` để chọn cách phù hợp.
5. Xử lý chuỗi trùng nhau và trường hợp `n = 1`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `strs`. Mọi chuỗi có cùng độ dài, chỉ gồm chữ thường, và **là hoán vị (anagram) của nhau**. `X` và `Y` tương tự nếu `X == Y` hoặc tồn tại `i ≠ j` sao cho đổi chỗ `X[i]`, `X[j]` thu được `Y`. Một nhóm là một thành phần liên thông theo quan hệ này. Trả về số nhóm.

**Ví dụ 1**

```text
Input: strs = ["tars","rats","arts","star"]
Output: 2
Giải thích: {"tars","rats","arts"} và {"star"}.
```

**Ví dụ 2**

```text
Input: strs = ["omv","ovm"]
Output: 1
```

**Ràng buộc**

- `1 <= strs.length <= 300`
- `1 <= strs[i].length <= 300`
- `strs[i]` chỉ gồm chữ thường.
- Mọi chuỗi trong `strs` có cùng độ dài và là anagram của nhau.

In [1]:
from typing import List

EXAMPLES = [
    (["tars", "rats", "arts", "star"], 2),
    (["omv", "ovm"], 1),
    (["abc"], 1),
    (["abc", "abc"], 1),                        # trùng nhau → tương tự
    (["abc", "bca", "cab"], 3),                 # mỗi cặp khác 3 vị trí
    (["abc", "bac", "bca", "cba", "acb", "cab"], 1),
]
for s, exp in EXAMPLES:
    print(s, "-> kỳ vọng", exp)

['tars', 'rats', 'arts', 'star'] -> kỳ vọng 2
['omv', 'ovm'] -> kỳ vọng 1
['abc'] -> kỳ vọng 1
['abc', 'abc'] -> kỳ vọng 1
['abc', 'bca', 'cab'] -> kỳ vọng 3
['abc', 'bac', 'bca', 'cba', 'acb', 'cab'] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Đếm vị trí khác nhau bằng `zip`

`zip(a, b)` ghép từng cặp ký tự cùng vị trí. Đếm số cặp khác nhau là `sum(x != y for x, y in zip(a, b))`.

In [2]:
a, b = "tars", "rats"
print(list(zip(a, b)))
print(sum(x != y for x, y in zip(a, b)))

[('t', 'r'), ('a', 'a'), ('r', 't'), ('s', 's')]
2


### 3.2. Kiểm tra "tương tự" khi hai chuỗi là anagram

Vì `a`, `b` là hoán vị của nhau, số vị trí khác nhau `d` không thể bằng 1 (nếu chỉ một vị trí khác thì đa tập ký tự khác nhau). Do đó:

- `d == 0`: bằng nhau → tương tự.
- `d == 2`: hai vị trí `i, j` khác; vì là anagram nên bắt buộc `a[i] == b[j]` và `a[j] == b[i]` → một lần đổi chỗ là đủ.
- `d >= 3`: không tương tự.

Ta có thể dừng sớm khi `d > 2`.

In [3]:
def similar(a, b):
    diff = 0
    for x, y in zip(a, b):
        if x != y:
            diff += 1
            if diff > 2:
                return False
    return True              # diff ∈ {0, 2} (diff = 1 không thể xảy ra với anagram)

print(similar("tars", "rats"), similar("tars", "arts"), similar("abc", "abc"))

True False True


### 3.3. Kiểm tra tổng quát (không giả thiết anagram)

Để an toàn khi làm biến thể, bản tổng quát kiểm tra đúng hai vị trí và chéo nhau.

In [4]:
def similar_general(a, b):
    if len(a) != len(b):
        return False
    idx = [i for i in range(len(a)) if a[i] != b[i]]
    if not idx:
        return True
    return len(idx) == 2 and a[idx[0]] == b[idx[1]] and a[idx[1]] == b[idx[0]]

print(similar_general("ab", "ba"), similar_general("ab", "ac"), similar_general("abc", "acd"))

True False False


### 3.4. Đếm thành phần liên thông bằng DFS

Lặp qua mọi đỉnh; gặp đỉnh chưa thăm thì tăng bộ đếm và DFS loang hết thành phần của nó.

In [5]:
def count_components(n, adj):
    seen = [False] * n
    comps = 0
    for s in range(n):
        if seen[s]:
            continue
        comps += 1
        stack = [s]
        seen[s] = True
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    stack.append(v)
    return comps

print(count_components(5, [[1], [0], [3], [2], []]))

3


### 3.5. Union-Find

Mỗi lần `union` thành công (hai gốc khác nhau) giảm số thành phần đi 1.

In [6]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n))
        self.r = [0] * n
        self.count = n

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.r[ra] < self.r[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]:
            self.r[ra] += 1
        self.count -= 1
        return True

d = DSU(4); d.union(0, 1); d.union(2, 3); d.union(1, 0)
print(d.count)

2


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Ma trận "tương tự" cho ví dụ 1

| | tars | rats | arts | star |
|---|---|---|---|---|
| tars | — | ✔ (t↔r) | ✗ (3 vị trí) | ✗ |
| rats | ✔ | — | ✔ (r↔a) | ✗ |
| arts | ✗ | ✔ | — | ✗ |
| star | ✗ | ✗ | ✗ | — |

Cạnh: `tars–rats`, `rats–arts`. Thành phần: `{tars, rats, arts}`, `{star}` → **2**.

### 4.2. Tính bắc cầu

`tars` và `arts` khác nhau ở 3 vị trí (không tương tự trực tiếp), nhưng cùng nhóm qua `rats`. Đó là lý do phải tìm **thành phần liên thông**, không chỉ đếm cặp.

### 4.3. Hai cách tìm cạnh

- **So mọi cặp:** `n(n−1)/2` cặp × `O(m)` → `O(n²m)`. Với `n·m <= 2·10⁴`, tệ nhất `n = 300, m ≈ 67` → `~3·10⁶` phép — ổn.
- **Sinh hàng xóm:** mỗi chuỗi có `m(m−1)/2` phép đổi chỗ, mỗi phép tạo chuỗi mới `O(m)` → `O(n·m³)`. Tốt khi `m` rất nhỏ và `n` rất lớn (biến thể).

In [7]:
strs1 = ["tars", "rats", "arts", "star"]
for i in range(4):
    print(f"{strs1[i]:5}", ["✔" if i != j and similar(strs1[i], strs1[j]) else "." for j in range(4)])

tars  ['.', '✔', '.', '.']
rats  ['✔', '.', '✔', '.']
arts  ['.', '✔', '.', '.']
star  ['.', '.', '.', '.']


## 5. Cách 1 — Brute Force: gộp nhóm lặp cho tới khi ổn định

**Ý tưởng (ngây thơ nhất).** Ban đầu mỗi chuỗi là một nhóm. Lặp: tìm hai nhóm có cặp phần tử tương tự → gộp lại. Dừng khi không gộp được nữa. Không cần biết DFS hay DSU.

In [8]:
def groups_brute(strs):
    groups = [[s] for s in strs]
    merged = True
    while merged:
        merged = False
        for i in range(len(groups)):
            for j in range(i + 1, len(groups)):
                if any(similar_general(a, b) for a in groups[i] for b in groups[j]):
                    groups[i] += groups[j]
                    del groups[j]
                    merged = True
                    break
            if merged:
                break
    return len(groups)

for s, exp in EXAMPLES:
    assert groups_brute(s) == exp
print("Brute force: OK")

Brute force: OK


**Độ phức tạp.** Mỗi lần gộp có thể phải quét lại mọi cặp nhóm và mọi cặp phần tử: `O(n²m)` mỗi lần, tối đa `n − 1` lần gộp → `O(n³m)`. Với `n = 300, m = 67`: ~`1.8·10⁹` — quá chậm.

**Nút thắt.** Ta kiểm tra **lại** các cặp đã biết là không tương tự sau mỗi lần gộp. Nên kiểm tra mỗi cặp **một lần**, rồi dùng cấu trúc đồ thị (DFS/BFS) hoặc DSU để gom.

## 6. Bước đột phá

**Insight 1 — Đồ thị ẩn.** Đỉnh `i` ↔ `strs[i]`; cạnh `i–j` nếu `similar(strs[i], strs[j])`. Đáp án = số thành phần liên thông. Mỗi cặp chỉ cần kiểm tra một lần.

**Insight 2 — Kiểm tra cặp `O(m)` với dừng sớm.** Nhờ giả thiết anagram, chỉ cần đếm vị trí khác nhau và dừng khi > 2.

**Insight 3 — Không cần dựng danh sách kề.** Trong DFS, khi đứng ở `u`, duyệt mọi `v` **chưa thăm** và kiểm tra `similar(u, v)` tại chỗ. Với DSU, chỉ kiểm tra cặp khi `find(i) != find(j)` — bỏ qua cặp đã cùng nhóm, tiết kiệm nhiều phép so sánh.

**Insight 4 — Sinh hàng xóm khi `m` nhỏ.** Đưa mọi chuỗi vào `set`; từ `u`, thử mọi phép đổi chỗ `(i, j)`, chuỗi kết quả có trong `set` chưa thăm → hàng xóm.

**Quy tắc nếu…thì:**
- *Nếu* hai chuỗi giống hệt *thì* cùng nhóm.
- *Nếu* `n²` nhỏ hơn `n·m²` (tức `n < m²`) *thì* so cặp; ngược lại cân nhắc sinh hàng xóm.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS trên đồ thị ẩn (so cặp)

In [9]:
def groups_dfs(strs):
    n = len(strs)
    seen = [False] * n
    groups = 0
    for s in range(n):
        if seen[s]:
            continue
        groups += 1
        seen[s] = True
        stack = [s]
        while stack:
            u = stack.pop()
            for v in range(n):
                if not seen[v] and similar(strs[u], strs[v]):
                    seen[v] = True
                    stack.append(v)
    return groups

for s, exp in EXAMPLES:
    assert groups_dfs(s) == exp
print("DFS đồ thị ẩn: OK")

DFS đồ thị ẩn: OK


Giải thích: mỗi đỉnh `u` được lấy ra khỏi stack đúng một lần và duyệt `n` đỉnh `v` (bỏ qua đỉnh đã thăm trước khi gọi `similar`) → tối đa `n²` lần gọi `similar`.

Code nộp LeetCode cho cách 2:

In [10]:
class SolutionDFS:
    def numSimilarGroups(self, strs: List[str]) -> int:
        def similar(a, b):
            diff = 0
            for x, y in zip(a, b):
                if x != y:
                    diff += 1
                    if diff > 2:
                        return False
            return True

        n = len(strs)
        seen = [False] * n
        groups = 0
        for s in range(n):
            if seen[s]:
                continue
            groups += 1
            seen[s] = True
            stack = [s]
            while stack:
                u = stack.pop()
                for v in range(n):
                    if not seen[v] and similar(strs[u], strs[v]):
                        seen[v] = True
                        stack.append(v)
        return groups

for s, exp in EXAMPLES:
    assert SolutionDFS().numSimilarGroups(s) == exp
print("SolutionDFS: OK")

SolutionDFS: OK


**Độ phức tạp cách 2:** `O(n²·m)` thời gian, `O(n)` bộ nhớ (không lưu cạnh).

### 7.2. Cách 3 — BFS trên đồ thị ẩn

Giống DFS, thay stack bằng `deque`. Một mẹo nhỏ: giữ danh sách `remaining` các chỉ số chưa thăm để không phải duyệt lại các đỉnh đã thăm.

In [11]:
from collections import deque

def groups_bfs(strs):
    remaining = set(range(len(strs)))
    groups = 0
    while remaining:
        s = remaining.pop()
        groups += 1
        q = deque([s])
        while q:
            u = q.popleft()
            nxt = [v for v in remaining if similar(strs[u], strs[v])]
            for v in nxt:
                remaining.discard(v)
                q.append(v)
    return groups

for s, exp in EXAMPLES:
    assert groups_bfs(s) == exp
print("BFS + tập chưa thăm: OK")

BFS + tập chưa thăm: OK


Code nộp LeetCode cho cách 3:

In [12]:
class SolutionBFS:
    def numSimilarGroups(self, strs: List[str]) -> int:
        def similar(a, b):
            diff = 0
            for x, y in zip(a, b):
                if x != y:
                    diff += 1
                    if diff > 2:
                        return False
            return True

        remaining = set(range(len(strs)))
        groups = 0
        while remaining:
            groups += 1
            q = deque([remaining.pop()])
            while q:
                u = q.popleft()
                nxt = [v for v in remaining if similar(strs[u], strs[v])]
                for v in nxt:
                    remaining.discard(v)
                    q.append(v)
        return groups

for s, exp in EXAMPLES:
    assert SolutionBFS().numSimilarGroups(s) == exp
print("SolutionBFS: OK")

SolutionBFS: OK


**Độ phức tạp cách 3:** `O(n²·m)` xấu nhất, thường nhanh hơn cách 2 vì tập `remaining` co dần.

### 7.3. Cách 4 — Union-Find (so cặp, bỏ qua cặp đã cùng nhóm)

In [13]:
def groups_dsu(strs):
    n = len(strs)
    dsu = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if dsu.find(i) != dsu.find(j) and similar(strs[i], strs[j]):
                dsu.union(i, j)
    return dsu.count

for s, exp in EXAMPLES:
    assert groups_dsu(s) == exp
print("Union-Find: OK")

Union-Find: OK


Kiểm tra `find(i) != find(j)` **trước** `similar` vì `find` gần `O(1)` còn `similar` là `O(m)`.

Code nộp LeetCode cho cách 4:

In [14]:
class SolutionDSU:
    def numSimilarGroups(self, strs: List[str]) -> int:
        n = len(strs)
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def similar(a, b):
            diff = 0
            for x, y in zip(a, b):
                if x != y:
                    diff += 1
                    if diff > 2:
                        return False
            return True

        groups = n
        for i in range(n):
            for j in range(i + 1, n):
                ri, rj = find(i), find(j)
                if ri != rj and similar(strs[i], strs[j]):
                    parent[ri] = rj
                    groups -= 1
        return groups

for s, exp in EXAMPLES:
    assert SolutionDSU().numSimilarGroups(s) == exp
print("SolutionDSU: OK")

SolutionDSU: OK


**Độ phức tạp cách 4:** `O(n²·(m + α(n)))` thời gian, `O(n)` bộ nhớ.

### 7.4. Cách 5 — Sinh hàng xóm bằng hoán đổi (khi `m` nhỏ)

Gom các chuỗi **khác nhau** vào `set` (chuỗi trùng tự động cùng nhóm). Từ chuỗi `u`, thử mọi cặp `(i, j)` với `u[i] != u[j]`, tạo chuỗi đổi chỗ, nếu nằm trong tập chưa thăm thì là hàng xóm.

In [15]:
def groups_swap(strs):
    unvisited = set(strs)
    groups = 0
    while unvisited:
        start = unvisited.pop()
        groups += 1
        stack = [start]
        while stack:
            u = stack.pop()
            chars = list(u)
            m = len(chars)
            for i in range(m):
                for j in range(i + 1, m):
                    if chars[i] == chars[j]:
                        continue
                    chars[i], chars[j] = chars[j], chars[i]
                    w = ''.join(chars)
                    chars[i], chars[j] = chars[j], chars[i]
                    if w in unvisited:
                        unvisited.remove(w)
                        stack.append(w)
    return groups

for s, exp in EXAMPLES:
    assert groups_swap(s) == exp
print("Sinh hàng xóm bằng hoán đổi: OK")

Sinh hàng xóm bằng hoán đổi: OK


Code nộp LeetCode cho cách 5 (lai: chọn chiến lược theo kích thước — so cặp nếu `n <= m²`, ngược lại sinh hàng xóm):

In [16]:
class SolutionHybrid:
    def numSimilarGroups(self, strs: List[str]) -> int:
        n, m = len(strs), len(strs[0])
        if n <= m * m:
            return SolutionDSU().numSimilarGroups(strs)
        unvisited = set(strs)
        groups = 0
        while unvisited:
            groups += 1
            stack = [unvisited.pop()]
            while stack:
                chars = list(stack.pop())
                for i in range(m):
                    for j in range(i + 1, m):
                        if chars[i] != chars[j]:
                            chars[i], chars[j] = chars[j], chars[i]
                            w = ''.join(chars)
                            chars[i], chars[j] = chars[j], chars[i]
                            if w in unvisited:
                                unvisited.remove(w)
                                stack.append(w)
        return groups

for s, exp in EXAMPLES:
    assert SolutionHybrid().numSimilarGroups(s) == exp
print("SolutionHybrid: OK")

SolutionHybrid: OK


**Độ phức tạp cách 5:** `O(k·m³)` với `k` = số chuỗi khác nhau (mỗi chuỗi sinh `m²/2` chuỗi dài `m`, băm `O(m)`). Bộ nhớ `O(k·m)`.

## 8. TRACE từng bước

### 8.1. Trace DSU trên ví dụ 1

In [17]:
def trace_dsu(strs):
    n = len(strs)
    dsu = DSU(n)
    calls = 0
    for i in range(n):
        for j in range(i + 1, n):
            if dsu.find(i) == dsu.find(j):
                print(f"({strs[i]}, {strs[j]}): đã cùng nhóm → bỏ qua, không gọi similar")
                continue
            calls += 1
            diff = [k for k in range(len(strs[i])) if strs[i][k] != strs[j][k]]
            ok = similar(strs[i], strs[j])
            if ok:
                dsu.union(i, j)
            print(f"({strs[i]}, {strs[j]}): vị trí khác {diff} → {'tương tự, union' if ok else 'không'}; số nhóm = {dsu.count}")
    print("số lần gọi similar:", calls, "| đáp án:", dsu.count)
    return dsu.count

trace_dsu(["tars", "rats", "arts", "star"])

(tars, rats): vị trí khác [0, 2] → tương tự, union; số nhóm = 3
(tars, arts): vị trí khác [0, 1, 2] → không; số nhóm = 3
(tars, star): vị trí khác [0, 1, 2, 3] → không; số nhóm = 3
(rats, arts): vị trí khác [0, 1] → tương tự, union; số nhóm = 2
(rats, star): vị trí khác [0, 1, 2, 3] → không; số nhóm = 2
(arts, star): vị trí khác [0, 1, 2, 3] → không; số nhóm = 2
số lần gọi similar: 6 | đáp án: 2


2

Bảng trạng thái:

| cặp | vị trí khác | hành động | số nhóm |
|---|---|---|---|
| (tars, rats) | [0, 2] | union | 3 |
| (tars, arts) | [0, 1, 2] | — | 3 |
| (tars, star) | [0,1,2,3] | — | 3 |
| (rats, arts) | [0, 1] | union | 2 |
| (rats, star) | … | — | 2 |
| (arts, star) | … | — | 2 |

### 8.2. Trace DFS — thứ tự loang

In [18]:
def trace_dfs(strs):
    n = len(strs)
    seen = [False] * n
    groups = 0
    for s in range(n):
        if seen[s]:
            continue
        groups += 1
        print(f"Nhóm {groups} bắt đầu từ '{strs[s]}'")
        seen[s] = True
        stack = [s]
        while stack:
            u = stack.pop()
            for v in range(n):
                if not seen[v] and similar(strs[u], strs[v]):
                    seen[v] = True
                    stack.append(v)
                    print(f"   '{strs[u]}' → '{strs[v]}'")
    return groups

trace_dfs(["abc", "bca", "cab", "bac", "abc", "acb"])

Nhóm 1 bắt đầu từ 'abc'
   'abc' → 'bac'
   'abc' → 'abc'
   'abc' → 'acb'
   'acb' → 'bca'
   'acb' → 'cab'


1

## 9. Code nộp LeetCode

Chọn cách 4 (Union-Find so cặp): ngắn, bỏ qua được nhiều phép so sánh, `O(n²m)` — với `n·m <= 2·10⁴` rất nhanh.

In [19]:
from typing import List

class Solution:
    def numSimilarGroups(self, strs: List[str]) -> int:
        n = len(strs)
        parent = list(range(n))

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]      # nén nửa đường
                x = parent[x]
            return x

        def similar(a: str, b: str) -> bool:
            diff = 0
            for x, y in zip(a, b):
                if x != y:
                    diff += 1
                    if diff > 2:
                        return False
            return True                            # 0 hoặc 2 vị trí khác (vì là anagram)

        groups = n
        for i in range(n):
            for j in range(i + 1, n):
                ri, rj = find(i), find(j)
                if ri != rj and similar(strs[i], strs[j]):
                    parent[ri] = rj
                    groups -= 1
        return groups

for s, exp in EXAMPLES:
    assert Solution().numSimilarGroups(s) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Hàm `similar`.** Giả sử `a`, `b` là anagram. Gọi `D` = tập vị trí khác nhau.

- `|D| = 0` ⇔ `a == b` → tương tự theo định nghĩa.
- `|D| = 1` không thể: ở các vị trí ngoài `D` hai chuỗi giống nhau, nên đa tập ký tự chỉ khác ở vị trí duy nhất đó → mâu thuẫn anagram.
- `|D| = 2`, `D = {i, j}`: đa tập `{a[i], a[j]} = {b[i], b[j]}` và `a[i] ≠ b[i]`, `a[j] ≠ b[j]` ⇒ `a[i] = b[j]`, `a[j] = b[i]` ⇒ đổi chỗ `i, j` trong `a` cho `b` → tương tự.
- `|D| >= 3`: một lần đổi chỗ thay đổi tối đa 2 vị trí → không tương tự.

Vậy `similar` trả `True` ⇔ `|D| <= 2` ⇔ tương tự. ✔

**Đếm nhóm.** Nhóm được định nghĩa là thành phần liên thông của đồ thị "tương tự". DFS/BFS từ một đỉnh thăm đúng thành phần của nó; mỗi lần bắt đầu từ đỉnh chưa thăm tạo một thành phần mới. DSU: bất biến "hai đỉnh cùng gốc ⇔ có đường đi qua các cạnh đã xét"; sau khi xét mọi cặp (bỏ qua cặp đã cùng gốc không làm mất thông tin vì cạnh đó không nối thêm gì), số gốc = số thành phần.

**Sinh hàng xóm (cách 5).** Hàng xóm của `u` trong đồ thị = các chuỗi trong `strs` nhận được từ `u` bằng một lần đổi chỗ — đúng tập được sinh ra (bỏ đổi chỗ hai ký tự giống nhau vì cho lại `u`). Chuỗi trùng nhau gộp làm một đỉnh vì chúng tương tự nhau.

## 11. Phân tích Time/Space

Gọi `n` = số chuỗi, `m` = độ dài, `n·m <= 2·10⁴`.

| Cách | Time | Space |
|---|---|---|
| 1. Gộp nhóm lặp | `O(n³·m)` | `O(n·m)` |
| 2. DFS so cặp | `O(n²·m)` | `O(n)` |
| 3. BFS + tập chưa thăm | `O(n²·m)` | `O(n)` |
| 4. Union-Find | `O(n²·m)` | `O(n)` |
| 5. Sinh hàng xóm | `O(n·m³)` | `O(n·m)` |

In [20]:
import random, time

def random_anagrams(n, m, swaps, rnd, alpha="abcdefghij"):
    base = [rnd.choice(alpha) for _ in range(m)]
    res = []
    for _ in range(n):
        s = base[:]
        for _ in range(rnd.randint(0, swaps)):
            i, j = rnd.randrange(m), rnd.randrange(m)
            s[i], s[j] = s[j], s[i]
        res.append(''.join(s))
    return res

rnd = random.Random(839)
case_a = random_anagrams(300, 66, 3, rnd)            # n lớn nhất, m vừa
case_b = random_anagrams(66, 300, 2, rnd)            # m lớn
for name, fn in [("dfs", groups_dfs), ("bfs", groups_bfs), ("dsu", groups_dsu),
                 ("swap", groups_swap), ("Solution", lambda x: Solution().numSimilarGroups(x))]:
    t0 = time.perf_counter()
    ra = fn(case_a)
    rb = fn(case_b) if name != "swap" else "bỏ qua (m=300 quá lớn)"
    print(f"{name:9}: {ra}, {rb}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

dfs      : 118, 29  (39 ms)


bfs      : 118, 29  (38 ms)


dsu      : 118, 29  (46 ms)


swap     : 118, bỏ qua (m=300 quá lớn)  (249 ms)
Solution : 118, 29  (44 ms)


Với `m = 300`, cách 5 phải sinh `~45 000` chuỗi dài 300 cho **mỗi** chuỗi — chậm hơn nhiều so với so cặp. Ngược lại với `m = 5`, `n = 10⁵` (biến thể), cách 5 thắng.

## 12. Bộ test

In [21]:
methods = {
    "brute": groups_brute,
    "dfs": groups_dfs,
    "bfs": groups_bfs,
    "dsu": groups_dsu,
    "swap": groups_swap,
    "SolutionDFS": lambda x: SolutionDFS().numSimilarGroups(x),
    "SolutionBFS": lambda x: SolutionBFS().numSimilarGroups(x),
    "SolutionDSU": lambda x: SolutionDSU().numSimilarGroups(x),
    "SolutionHybrid": lambda x: SolutionHybrid().numSimilarGroups(x),
    "Solution": lambda x: Solution().numSimilarGroups(x),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    (["a"], 1),
    (["a", "a", "a"], 1),
    (["aab", "aba", "baa"], 1),
    (["kccomwcgcs", "socgcmcwkc", "sgckwcmcoc", "coswcmcgkc", "cowkccmsgc", "cosgmccwkc",
      "sgmkwcccoc", "coswmccgkc", "kowcccmsgc", "kgcomwcccs"], 5),
]
for name, fn in methods.items():
    for s, exp in edge:
        assert fn(s) == exp, (name, s, fn(s), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: đối chiếu brute force (dùng similar_general)
for seed in range(400):
    r_ = random.Random(seed)
    n = r_.randint(1, 12)
    m = r_.randint(1, 6)
    strs = random_anagrams(n, m, r_.randint(0, 4), r_, alpha="abc"[: r_.randint(1, 3)])
    ref = groups_brute(strs)
    for name, fn in methods.items():
        assert fn(strs) == ref, (name, strs)
print("400 bộ ngẫu nhiên: khớp brute force")

# 12.3 Dữ liệu vừa: các cách nhanh khớp nhau
for seed in range(10):
    r_ = random.Random(900 + seed)
    strs = random_anagrams(120, 8, 3, r_)
    vals = {name: fn(strs) for name, fn in methods.items() if name != "brute"}
    assert len(set(vals.values())) == 1, vals
print("Dữ liệu vừa: các cách nhanh khớp nhau")

Ví dụ + edge cases: OK
400 bộ ngẫu nhiên: khớp brute force
Dữ liệu vừa: các cách nhanh khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm số cặp tương tự thay vì số nhóm

**Code sai:**

In [22]:
def wrong_count_pairs(strs):
    n = len(strs)
    linked = sum(1 for i in range(n) for j in range(i + 1, n) if similar(strs[i], strs[j]))
    return n - linked                                      # SAI: coi mỗi cặp giảm 1 nhóm

s_bad = ["abc", "acb", "bac", "bca", "cab", "cba"]       # 6 hoán vị: 9 cặp tương tự, 1 nhóm
print("sai:", wrong_count_pairs(s_bad), " đúng:", Solution().numSimilarGroups(s_bad))

sai: -3  đúng: 1


**Vì sao sai:** nếu các cạnh tạo chu trình (hoặc nhiều cạnh trong cùng nhóm), mỗi cạnh thừa vẫn bị trừ → kết quả quá nhỏ, có khi âm. **Code đúng:** chỉ trừ khi `union` thật sự nối hai nhóm khác nhau.

### Lỗi 2 — Kiểm tra "tương tự" chỉ bằng `diff == 2` (quên chuỗi trùng)

**Code sai:**

In [23]:
def wrong_similar_eq2(a, b):
    return sum(x != y for x, y in zip(a, b)) == 2           # SAI: a == b cũng là tương tự

def groups_with(sim, strs):
    n = len(strs); d = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if sim(strs[i], strs[j]):
                d.union(i, j)
    return d.count

print("sai:", groups_with(wrong_similar_eq2, ["abc", "abc"]), " đúng:", Solution().numSimilarGroups(["abc", "abc"]))

sai: 2  đúng: 1


**Vì sao sai:** đề nói "bằng nhau" cũng là tương tự. Hai chuỗi trùng phải cùng nhóm. **Code đúng:** `diff <= 2` (với anagram), hoặc `diff == 0 or diff == 2`.

### Lỗi 3 — Dùng `similar` cho chuỗi không phải anagram

Hàm `similar` "đếm `diff <= 2`" chỉ đúng nhờ giả thiết anagram. Nếu dùng cho biến thể tổng quát, `"ab"` và `"cd"` (khác 2 vị trí) bị coi là tương tự.

In [24]:
print("similar('ab','cd') =", similar("ab", "cd"), "| tổng quát:", similar_general("ab", "cd"))

similar('ab','cd') = True | tổng quát: False


**Code đúng cho biến thể:** kiểm tra chéo `a[i] == b[j] and a[j] == b[i]` (hàm `similar_general`).

### Lỗi 4 — DFS đánh dấu khi lấy ra (không phải khi đẩy vào)

Không sai kết quả nhưng một đỉnh có thể bị đẩy vào stack nhiều lần → nhiều lần gọi `similar` thừa, có thể gấp `n` lần.

In [25]:
def dfs_mark_on_pop(strs):
    n = len(strs); seen = [False] * n; groups = 0; pushes = 0
    for s in range(n):
        if seen[s]:
            continue
        groups += 1
        stack = [s]
        while stack:
            u = stack.pop()
            if seen[u]:
                continue
            seen[u] = True
            for v in range(n):
                if not seen[v] and similar(strs[u], strs[v]):
                    stack.append(v); pushes += 1
    return groups, pushes

same = ["abcd"] * 60
print("đánh dấu khi pop: (nhóm, số lần push) =", dfs_mark_on_pop(same))

đánh dấu khi pop: (nhóm, số lần push) = (1, 1770)


**Code đúng:** đánh dấu `seen[v] = True` ngay khi đẩy vào stack (như cách 2) → mỗi đỉnh được đẩy đúng một lần.

## 14. Mẹo & nhận dạng pattern

- **"Nhóm theo quan hệ lan truyền (bắc cầu)"** → thành phần liên thông: DFS/BFS hoặc DSU.
- **Đồ thị ẩn:** không có danh sách cạnh → hoặc so mọi cặp (`O(n²·chi phí so)`), hoặc sinh hàng xóm (`O(n·số hàng xóm·chi phí sinh)`). Chọn theo kích thước.
- **Tận dụng giả thiết đề** (anagram) để đơn giản hoá phép kiểm tra cạnh.
- **DSU + kiểm tra `find` trước phép so đắt** là mẹo tiết kiệm thời gian.
- Bài liên quan: **547 Number of Provinces**, **721 Accounts Merge**, **952 Largest Component Size by Common Factor**, **127 Word Ladder** (sinh hàng xóm), **854 K-Similar Strings**.

## 15. Biến thể

1. **Không giả thiết anagram**: dùng `similar_general`.
2. **Trả về các nhóm** (danh sách chuỗi) thay vì số lượng: gom theo `find(i)`.
3. **Tương tự = khác nhau tối đa 1 ký tự** (thay thế): giống Word Ladder; sinh hàng xóm bằng mẫu `"ab*d"`.
4. **`n` rất lớn, `m` nhỏ** (`n = 10⁵`, `m = 6`): cách 5 sinh hàng xóm.
5. **Kích thước nhóm lớn nhất**: DSU với mảng `size`.

Minh hoạ biến thể 2 và 5:

In [26]:
def similar_groups_list(strs):
    n = len(strs)
    d = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if d.find(i) != d.find(j) and similar_general(strs[i], strs[j]):
                d.union(i, j)
    groups = {}
    for i in range(n):
        groups.setdefault(d.find(i), []).append(strs[i])
    return sorted(groups.values(), key=len, reverse=True)

g = similar_groups_list(["tars", "rats", "arts", "star"])
print(g, "| nhóm lớn nhất:", len(g[0]))

[['tars', 'rats', 'arts'], ['star']] | nhóm lớn nhất: 3


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Gộp nhóm lặp | gộp hai nhóm có cặp tương tự | `O(n³m)` | `O(nm)` | đối chiếu |
| 2. DFS so cặp | đồ thị ẩn, đánh dấu khi đẩy | `O(n²m)` | `O(n)` | dễ viết |
| 3. BFS + tập chưa thăm | chỉ so với đỉnh còn lại | `O(n²m)` | `O(n)` | nhanh hơn thực tế |
| 4. Union-Find | bỏ qua cặp đã cùng nhóm | `O(n²m)` | `O(n)` | **nộp bài** |
| 5. Sinh hàng xóm | mọi phép đổi chỗ + `set` | `O(nm³)` | `O(nm)` | `m` nhỏ, `n` lớn |

## 17. Bài tập

**Bài 1 (đoán output).** `["abc", "acb", "bac", "bca", "cab", "cba"]` có bao nhiêu nhóm? `["abcd", "badc"]`?

**Bài 2 (tính tay).** Vẽ đồ thị tương tự cho `["kccomwcgcs","socgcmcwkc","sgckwcmcoc","coswcmcgkc","cowkccmsgc","cosgmccwkc","sgmkwcccoc","coswmccgkc","kowcccmsgc","kgcomwcccs"]` (gợi ý: dùng code in ma trận như mục 4) và xác định 5 nhóm.

**Bài 3 (điền TODO).** Viết `largest_group(strs)` trả về kích thước nhóm lớn nhất bằng DSU có `size`.

**Bài 4 (tự viết lại).** Viết lại `Solution` bằng DFS **đệ quy**, giải thích vì sao độ sâu đệ quy tối đa `n = 300` là an toàn.

**Bài 5 (mini challenge).** Biến thể 3: tương tự = khác **đúng một** ký tự (cùng độ dài, không cần anagram). Đếm nhóm bằng mẫu ký tự đại diện `*`.

In [27]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def largest_group_todo(strs):
    # TODO: DSU với size; union khi similar_general; trả về max size của gốc
    pass

# Bài 4 — TODO
def solution_recursive_todo(strs):
    # TODO: def dfs(u): seen[u] = True; for v...: if not seen[v] and similar: dfs(v)
    pass

# Bài 5 — TODO
def groups_one_char_todo(words):
    # TODO: bucket[pattern] = [chỉ số]; pattern = w[:i] + '*' + w[i+1:]; union mọi chỉ số cùng bucket
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: hai hoán vị của 3 ký tự khác nhau ở 2 hoặc 3 vị trí.
- Mức 2: mỗi hoán vị nối với 3 hoán vị khác (các phép đổi chỗ) → liên thông.
- Mức 3: `1`; `["abcd","badc"]` khác 4 vị trí → `2`.

**Bài 2.**
- Mức 1: tính ma trận `similar`.
- Mức 2: chạy `similar_groups_list`.
- Mức 3: 5 nhóm (kết quả đề kiểm tra ở bộ test).

**Bài 3.**
- Mức 1: thêm mảng `size` vào DSU.
- Mức 2: khi gộp, `size[gốc mới] += size[gốc cũ]`.
- Mức 3: `max(size[find(i)] for i)`.

**Bài 4.**
- Mức 1: `seen` ngoài hàm.
- Mức 2: gọi `dfs(s)` cho mỗi `s` chưa thăm, đếm.
- Mức 3: độ sâu ≤ `n = 300` < 1000.

**Bài 5.**
- Mức 1: hai từ khác đúng 1 vị trí `i` ⇔ cùng mẫu với `*` ở vị trí `i`.
- Mức 2: `defaultdict(list)` theo mẫu.
- Mức 3: union phần tử đầu của mỗi bucket với các phần tử còn lại.

## 19. Đáp án tham khảo

In [28]:
from collections import defaultdict

# Bài 1
assert Solution().numSimilarGroups(["abc", "acb", "bac", "bca", "cab", "cba"]) == 1
assert Solution().numSimilarGroups(["abcd", "badc"]) == 2

# Bài 2
big10 = ["kccomwcgcs", "socgcmcwkc", "sgckwcmcoc", "coswcmcgkc", "cowkccmsgc", "cosgmccwkc",
         "sgmkwcccoc", "coswmccgkc", "kowcccmsgc", "kgcomwcccs"]
for grp in similar_groups_list(big10):
    print(grp)

# Bài 3
def largest_group(strs):
    n = len(strs)
    parent, size = list(range(n)), [1] * n
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in range(n):
        for j in range(i + 1, n):
            ri, rj = find(i), find(j)
            if ri != rj and similar_general(strs[i], strs[j]):
                if size[ri] < size[rj]:
                    ri, rj = rj, ri
                parent[rj] = ri; size[ri] += size[rj]
    return max(size[find(i)] for i in range(n))

assert largest_group(["tars", "rats", "arts", "star"]) == 3

# Bài 4
def solution_recursive(strs):
    n = len(strs); seen = [False] * n
    def dfs(u):
        seen[u] = True
        for v in range(n):
            if not seen[v] and similar(strs[u], strs[v]):
                dfs(v)
    groups = 0
    for s in range(n):
        if not seen[s]:
            groups += 1; dfs(s)
    return groups

for s, exp in EXAMPLES:
    assert solution_recursive(s) == exp

# Bài 5
def groups_one_char(words):
    n = len(words)
    d = DSU(n)
    bucket = defaultdict(list)
    for idx, w in enumerate(words):
        for i in range(len(w)):
            bucket[w[:i] + '*' + w[i + 1:]].append(idx)
    for ids in bucket.values():
        for k in ids[1:]:
            d.union(ids[0], k)
    return d.count

def groups_one_char_brute(words):
    def ok(a, b):
        return a == b or sum(x != y for x, y in zip(a, b)) == 1
    n = len(words); d = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if ok(words[i], words[j]):
                d.union(i, j)
    return d.count

for seed in range(200):
    r_ = random.Random(seed)
    words = [''.join(r_.choice("ab") for _ in range(3)) for _ in range(r_.randint(1, 8))]
    assert groups_one_char(words) == groups_one_char_brute(words)
print(groups_one_char(["hot", "dot", "dog", "cog", "abc"]))
print("Tất cả đáp án đúng")

['coswcmcgkc', 'cosgmccwkc', 'coswmccgkc']
['kccomwcgcs', 'kgcomwcccs']
['sgckwcmcoc', 'sgmkwcccoc']
['cowkccmsgc', 'kowcccmsgc']
['socgcmcwkc']
2
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (DSU trên đồ thị ẩn):**

```python
parent = list(range(n)); groups = n
for i in range(n):
    for j in range(i + 1, n):
        ri, rj = find(i), find(j)
        if ri != rj and similar(strs[i], strs[j]):   # find trước, similar sau
            parent[ri] = rj; groups -= 1
return groups
```

**`similar` cho anagram:** đếm vị trí khác, dừng khi `> 2`, trả `True` nếu `<= 2`.

**Checklist trước khi submit:**

- [ ] Chuỗi **giống hệt** là tương tự (`diff == 0`).
- [ ] Chỉ giảm số nhóm khi `union` nối hai gốc **khác** nhau.
- [ ] Kiểm tra `find` trước phép so `O(m)`.
- [ ] Nhớ giả thiết anagram; biến thể tổng quát phải kiểm tra chéo.
- [ ] DFS đánh dấu khi **đẩy vào** stack.